# CLIP Monte Carlo Dropout Embedding Explorer

This notebook loads every image under `data/notebook`, computes deterministic CLIP embeddings,
then runs Monte Carlo Dropout (MCDO) sampling to capture embedding uncertainty. The combined
embeddings are projected to two dimensions with PCA and visualised with per-image scatter plots,
mean markers, and covariance ellipses. Configure the paths and dropout settings in the next cell
to suit your workspace.

In [1]:
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

from uclip.core.utils import load_clip_backbone, set_determinism
from uclip.core.dropout import insert_adapters, override_dropout_rate, dump_dropout_rates
from uclip.core.sampling import sample_embeddings, compute_embedding_statistics

## Configuration

In [2]:
DATA_DIR = Path("data/notebook")  # Folder containing the images to analyse
MODEL_ID = "openai/clip-vit-base-patch32"
DEVICE = None  # Set to e.g. "cuda" to override automatic device selection
MC_PASSES = 32  # Number of stochastic forward passes for MCDO
MICROBATCH = 4  # Microbatch size for sampling helper
SEED = 0

# Optional dropout instrumentation. Populate ADAPTER_TARGETS to insert DropoutAdapters
# (see cli scripts for examples). OVERRIDE_DROPOUT_RATE changes existing dropout layers.
ADAPTER_TARGETS = []
ADAPTER_DROPOUT_P = 0.1
OVERRIDE_DROPOUT_RATE = None

In [3]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
if not DATA_DIR.exists():
    raise FileNotFoundError(f"DATA_DIR {DATA_DIR} does not exist. Populate it with images first.")

image_paths = sorted(
    path for path in DATA_DIR.rglob('*')
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)
if not image_paths:
    raise RuntimeError(f"No supported image files found under {DATA_DIR}.")

print(f"Found {len(image_paths)} images")
for path in image_paths:
    print(" -", path.relative_to(DATA_DIR))

FileNotFoundError: DATA_DIR data/notebook does not exist. Populate it with images first.

In [4]:
set_determinism(SEED)

loaded = load_clip_backbone(MODEL_ID, device=DEVICE)
model, processor, device = loaded.model, loaded.processor, loaded.device
model.eval()

if ADAPTER_TARGETS:
    insert_adapters(model, ADAPTER_TARGETS, p=ADAPTER_DROPOUT_P)
if OVERRIDE_DROPOUT_RATE is not None:
    override_dropout_rate(model, OVERRIDE_DROPOUT_RATE)

count, rates = dump_dropout_rates(model)
print(f"Using device: {device}")
print(f"Detected {count} dropout layers")
if rates:
    print(f"Dropout rates: {rates}")

RuntimeError: Failed to import transformers.models.clip.image_processing_clip_fast because of the following error (look up to see its traceback):
invalid syntax (<string>, line 1)

In [ ]:
records = []
for path in image_paths:
    with Image.open(path) as handle:
        image = handle.convert("RGB")
    inputs = processor(images=image, return_tensors="pt")
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        deterministic = model.get_image_features(**inputs).squeeze(0)

    samples = sample_embeddings(
        model=model,
        forward_fn=model.get_image_features,
        forward_kwargs=inputs,
        passes=MC_PASSES,
        microbatch=MICROBATCH,
    )
    stats = compute_embedding_statistics(samples)

    # Return model to eval mode after toggling dropout modules during sampling.
    model.eval()

    records.append(
        {
            "path": path,
            "deterministic": deterministic.detach().cpu().numpy(),
            "mc_samples": stats.embeddings.detach().cpu().numpy(),
            "mc_mean": stats.mean.detach().cpu().numpy(),
        }
    )

print("Finished encoding " + str(len(records)) + " images")
print("Total stochastic samples: " + str(sum(r["mc_samples"].shape[0] for r in records)))

In [ ]:
def fit_pca(matrix: np.ndarray, n_components: int = 2):
    if matrix.ndim != 2:
        raise ValueError("matrix must be 2D")
    if matrix.shape[0] < 2:
        raise ValueError("Need at least two samples to run PCA")
    mean = matrix.mean(axis=0)
    centered = matrix - mean
    u, s, vt = np.linalg.svd(centered, full_matrices=False)
    components = vt[:n_components]
    explained_variance = (s[:n_components] ** 2) / max(1, matrix.shape[0] - 1)
    total_variance = (s ** 2).sum() / max(1, matrix.shape[0] - 1)
    explained_ratio = explained_variance / total_variance if total_variance > 0 else np.zeros_like(explained_variance)
    return mean, components, explained_variance, explained_ratio

all_samples = np.concatenate([record["mc_samples"] for record in records], axis=0)

global_mean, components, explained_var, explained_ratio = fit_pca(all_samples, n_components=2)
for record in records:
    record["proj_samples"] = (record["mc_samples"] - global_mean) @ components.T
    record["proj_mean"] = (record["mc_mean"] - global_mean) @ components.T
    record["proj_deterministic"] = (record["deterministic"] - global_mean) @ components.T
    if record["proj_samples"].shape[0] < 2:
        record["proj_cov"] = np.zeros((2, 2))
    else:
        record["proj_cov"] = np.cov(record["proj_samples"], rowvar=False)

print("Explained variance ratios:", explained_ratio)

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Ellipse, Patch

fig, ax = plt.subplots(figsize=(8, 6))
colors = plt.cm.tab10(np.linspace(0, 1, len(records)))

def plot_covariance_ellipse(ax, mean, cov, color, n_std=1.0, alpha=0.2):
    if cov.shape != (2, 2):
        raise ValueError("Covariance must be 2x2 for ellipse plotting")
    eigvals, eigvecs = np.linalg.eigh(cov)
    eigvals = np.clip(eigvals, a_min=0.0, a_max=None)
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]
    width = 2 * n_std * np.sqrt(eigvals[0])
    height = 2 * n_std * np.sqrt(eigvals[1])
    angle = np.degrees(np.arctan2(eigvecs[1, 0], eigvecs[0, 0]))
    ellipse = Ellipse(xy=mean, width=width, height=height, angle=angle, facecolor=color, edgecolor=color, alpha=alpha)
    ax.add_patch(ellipse)

for idx, record in enumerate(records):
    color = colors[idx % len(colors)]
    samples_2d = record["proj_samples"]
    det_point = record["proj_deterministic"]
    mean_point = record["proj_mean"]

    ax.scatter(samples_2d[:, 0], samples_2d[:, 1], color=color, marker="x", alpha=0.65)
    ax.scatter(det_point[0], det_point[1], color=color, marker="o", edgecolor="white", s=80, zorder=3)
    ax.scatter(mean_point[0], mean_point[1], color=color, marker="*", edgecolor="black", s=140, zorder=4)
    plot_covariance_ellipse(ax, mean_point, record["proj_cov"], color=color, alpha=0.18)
    ax.text(mean_point[0], mean_point[1], record["path"].stem, color=color, fontsize=9, ha="center", va="center", zorder=5)

legend_handles = [
    Line2D([0], [0], marker="x", color="gray", linestyle="", label="MCDO samples"),
    Line2D([0], [0], marker="o", color="gray", markerfacecolor="gray", markeredgecolor="white", linestyle="", label="Deterministic CLIP"),
    Line2D([0], [0], marker="*", color="gray", markerfacecolor="gray", markeredgecolor="black", linestyle="", label="MCDO mean"),
    Patch(facecolor="gray", edgecolor="gray", alpha=0.18, label="1σ ellipse"),
]
ax.legend(handles=legend_handles, loc="upper right")
ax.set_xlabel(f"PC1 ({explained_ratio[0] * 100:.1f}% var)")
ax.set_ylabel(f"PC2 ({explained_ratio[1] * 100:.1f}% var)")
ax.set_title("CLIP embedding geometry under Monte Carlo Dropout")
ax.axhline(0, color="black", linewidth=0.5, alpha=0.3)
ax.axvline(0, color="black", linewidth=0.5, alpha=0.3)
fig.tight_layout()
plt.show()

### Next steps
- Add more images to `data/notebook` to enlarge the PCA basis.
- Adjust `ADAPTER_TARGETS` or `OVERRIDE_DROPOUT_RATE` if you want to instrument additional dropout.
- Extend the workflow to compare different CLIP checkpoints or apply custom perturbations before encoding.